# 🌳 Random Forest — Applied on Real Kaggle Datasets
### Notebook 2 of 2

---

## 👋 Welcome!

In **Notebook 1**, we learned all Random Forest concepts using synthetic (made-up) data.
In **this notebook**, we apply the **exact same concepts on real datasets**, just like a data scientist would at a company!

---

## 📚 Datasets Used

| Task | Dataset | Source |
|------|---------|--------|
| **Classification** | Titanic — Survival Prediction | seaborn built-in (same as Kaggle Titanic) |
| **Regression** | California Housing — House Price Prediction | sklearn built-in |

---

## 📋 What You Will Practice

| Step | Task |
|------|------|
| 1 | Real data loading |
| 2 | Real EDA — finding patterns in messy data |
| 3 | Handling missing values and encoding |
| 4 | Random Forest Classification on Titanic |
| 5 | Hyperparameter tuning |
| 6 | Full evaluation |
| 7 | Random Forest Regression on California Housing |
| 8 | Full regression evaluation |
| 9 | 🎓 Capstone Mini-Project |

---

> 💡 **Reminder**: Every new term is explained before the code cell that uses it!

In [ ]:
# ═══════════════════════════════════════════════════
# 📦  IMPORTING LIBRARIES
# ═══════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score, roc_curve, auc,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import fetch_california_housing

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print("✅ All libraries imported!")

---

# 🚢 Part 1: Classification — Titanic Survival Prediction

---

### 🎯 Problem Statement
> Predict whether a passenger **survived** (1) or **did not survive** (0) the Titanic disaster.

### 📊 About the Dataset
The Titanic dataset is one of the most famous datasets in Machine Learning.
It contains information about 891 passengers on the RMS Titanic.

| Column | Description |
|--------|-------------|
| `survived` | **Target** — 0 = Did not survive, 1 = Survived |
| `pclass` | Passenger class (1 = First, 2 = Second, 3 = Third) |
| `sex` | Gender (male / female) |
| `age` | Age in years |
| `sibsp` | # of siblings/spouses aboard |
| `parch` | # of parents/children aboard |
| `fare` | Ticket fare |
| `embarked` | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |
| `alone` | Was the passenger travelling alone? |

---

> 💡 **Learning Goal**: Handle missing values, encode categorical columns, and build a classifier on real messy data!

## 📂 Step 1: Load the Titanic Dataset

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **Categorical** | A column with text values (like 'male'/'female', 'C'/'Q'/'S') |
| **Numerical** | A column with numbers (like age, fare) |
| **seaborn built-in** | Datasets included with the seaborn library — no download needed! |

In [ ]:
# ═══════════════════════════════════════════════════
# 📂  STEP 1 — LOAD DATA
# ═══════════════════════════════════════════════════

# Load from seaborn — same as the Kaggle Titanic training set
titanic = sns.load_dataset('titanic')

print("✅ Titanic dataset loaded!")
print(f"Shape: {titanic.shape}")
print(f"\nAll columns: {list(titanic.columns)}")
print("\nFirst 5 rows:")
display(titanic.head())

## 🔍 Step 2: Exploratory Data Analysis (EDA)

---

### 🎯 What We'll Explore
1. Missing values — which columns have null/empty values?
2. Survival rate overall and by key features
3. Distributions of important features

---

> 🎯 **What we learn here**: Real data has missing values and messy columns. EDA helps us decide how to fix them.

In [ ]:
# ─── Missing Values ──────────────────────────────────────────
print("❓ Missing Values:")
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(1)
mv_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
mv_df = mv_df[mv_df['Missing Count'] > 0]
print(mv_df)

# Visualise
plt.figure(figsize=(8, 4))
mv_df['Missing %'].plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Missing Values by Column (%)', fontsize=12, fontweight='bold')
plt.ylabel('Missing %'); plt.xlabel('Column')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

print("\n💡 Strategy:")
print("   age      → Fill with MEDIAN (not mean — robust to outliers)")
print("   embarked → Fill with MODE (most frequent value)")
print("   deck     → Drop (>77% missing — too much to impute)")

In [ ]:
# ─── Survival Rate Overall ────────────────────────────────────
print(f"Overall Survival Rate: {titanic['survived'].mean()*100:.1f}%")
print(titanic['survived'].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# By gender
survival_sex = titanic.groupby('sex')['survived'].mean() * 100
survival_sex.plot(kind='bar', ax=axes[0], color=['lightblue','lightcoral'],
                  edgecolor='black', width=0.5)
axes[0].set_title('Survival Rate by Gender', fontweight='bold')
axes[0].set_ylabel('Survival Rate (%)'); axes[0].set_xticklabels(['Female','Male'], rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%', (p.get_x()+p.get_width()/2, p.get_height()+1),
                     ha='center', fontsize=11)

# By class
survival_class = titanic.groupby('pclass')['survived'].mean() * 100
survival_class.plot(kind='bar', ax=axes[1], color=['gold','silver','sienna'],
                    edgecolor='black', width=0.5)
axes[1].set_title('Survival Rate by Passenger Class', fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)'); axes[1].set_xticklabels(['1st','2nd','3rd'], rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x()+p.get_width()/2, p.get_height()+1),
                     ha='center', fontsize=11)

# Age distribution
titanic[titanic['survived']==1]['age'].dropna().hist(ax=axes[2], bins=20, alpha=0.6,
    label='Survived', color='green')
titanic[titanic['survived']==0]['age'].dropna().hist(ax=axes[2], bins=20, alpha=0.6,
    label='Did not survive', color='red')
axes[2].set_title('Age Distribution', fontweight='bold')
axes[2].set_xlabel('Age'); axes[2].set_ylabel('Count'); axes[2].legend()

plt.suptitle('Titanic EDA — Survival Patterns', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Fare Distribution by Class ──────────────────────────────
plt.figure(figsize=(10, 5))
for cls, color, lbl in zip([1,2,3], ['gold','silver','sienna'], ['1st','2nd','3rd']):
    titanic[titanic['pclass']==cls]['fare'].hist(bins=30, alpha=0.5,
        label=f'Class {cls} ({lbl})', color=color)
plt.xlabel('Fare (£)'); plt.ylabel('Count')
plt.title('Fare Distribution by Passenger Class', fontsize=12, fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

# Correlation heatmap on numerical columns
plt.figure(figsize=(8, 5))
num_cols = ['survived','pclass','age','sibsp','parch','fare']
sns.heatmap(titanic[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation — Numerical Columns', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 🔧 Step 3: Data Preprocessing

---

### 🎯 What We Need to Do

| Problem | Solution |
|---------|----------|
| `age` has missing values (19.9%) | Fill with **median age** |
| `embarked` has 2 missing values | Fill with **mode** (most frequent port) |
| `deck` has 77% missing | **Drop** the column |
| `sex` is text (male/female) | **Encode** to numbers (0/1) |
| `embarked` is text (C/Q/S) | **Encode** to numbers |
| Many redundant columns | **Drop** unnecessary ones |

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **Imputation** | Filling in missing values with a calculated substitute |
| **Median** | The middle value when sorted — good for skewed data |
| **Label Encoding** | Converting text categories to numbers (male→1, female→0) |
| **One-Hot Encoding** | Creating a new 0/1 column for each category |

---

> 💡 **Important**: Always fill missing values BEFORE splitting into train/test!

In [ ]:
# ═══════════════════════════════════════════════════
# 🔧  STEP 3 — PREPROCESSING
# ═══════════════════════════════════════════════════

df_t = titanic.copy()   # Work on a copy so original is unchanged

# ── 1. Drop columns we don't need ──────────────────
# deck     : too many missing values
# embark_town, alive, who, adult_male : redundant/derived columns
drop_cols = ['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class']
df_t.drop(columns=drop_cols, inplace=True)

print(f"After dropping: {list(df_t.columns)}")

# ── 2. Fill missing values ─────────────────────────
df_t['age'].fillna(df_t['age'].median(), inplace=True)
df_t['embarked'].fillna(df_t['embarked'].mode()[0], inplace=True)

# ── 3. Encode categorical columns ─────────────────
# sex: female→0, male→1
df_t['sex'] = LabelEncoder().fit_transform(df_t['sex'])

# embarked: C→0, Q→1, S→2
df_t['embarked'] = LabelEncoder().fit_transform(df_t['embarked'].astype(str))

# alone: True→1, False→0 (already boolean, but make sure it's int)
df_t['alone'] = df_t['alone'].astype(int)

print("\n✅ Preprocessing complete!")
print(f"Missing values remaining: {df_t.isnull().sum().sum()}")
print("\nFinal dataset preview:")
display(df_t.head())

In [ ]:
# ─── Train-Test Split ─────────────────────────────────────────
X_t = df_t.drop('survived', axis=1)
y_t = df_t['survived']

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)

print(f"Features used: {list(X_t.columns)}")
print(f"\nTrain: {X_tr_t.shape[0]} passengers | Test: {X_te_t.shape[0]} passengers")
print(f"Train survival rate: {y_tr_t.mean()*100:.1f}%")
print(f"Test  survival rate: {y_te_t.mean()*100:.1f}%")

## 🌳 Step 4: Build Random Forest Classifier

---

> 🎯 **What we learn here**: Apply the same classifier we built in Notebook 1 on a real dataset with 7 features.

In [ ]:
# ═══════════════════════════════════════════════════
# 🌳  TRAIN RANDOM FOREST — TITANIC
# ═══════════════════════════════════════════════════

rf_t = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_t.fit(X_tr_t, y_tr_t)
y_pred_t = rf_t.predict(X_te_t)

print("=" * 55)
print("TITANIC — RANDOM FOREST CLASSIFIER")
print("=" * 55)
print(f"Train Accuracy : {rf_t.score(X_tr_t, y_tr_t)*100:.2f}%")
print(f"Test  Accuracy : {accuracy_score(y_te_t, y_pred_t)*100:.2f}%")
print(f"OOB   Score    : {rf_t.oob_score_*100:.2f}%  ← free validation!")
print("=" * 55)

## 📏 Step 5: Full Model Evaluation

---

### Confusion Matrix — Both Ways
We always show BOTH the old way (Seaborn) and the new way (ConfusionMatrixDisplay)
so you are comfortable with both in interviews and industry work.

---

> 🎯 **What we learn here**: Complete evaluation pipeline on a real classification problem.

In [ ]:
# ─── Confusion Matrix OLD Way ─────────────────────────────────
cm_t = confusion_matrix(y_te_t, y_pred_t)
print("Confusion Matrix (array):")
print(cm_t)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: No','Predicted: Yes'],
            yticklabels=['Actual: No','Actual: Yes'],
            linewidths=1)
plt.title('Confusion Matrix — Old Way (Seaborn)\nTitanic Survival', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Confusion Matrix NEW Way ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_te_t, y_pred_t,
    display_labels=['Not Survived','Survived'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('from_predictions()\n(standard)', fontweight='bold')

ConfusionMatrixDisplay.from_estimator(
    rf_t, X_te_t, y_te_t,
    display_labels=['Not Survived','Survived'],
    cmap='Oranges', ax=axes[1]
)
axes[1].set_title('from_estimator()\n(directly from model)', fontweight='bold')

plt.suptitle('Confusion Matrix — New Way (ConfusionMatrixDisplay)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Classification Report ────────────────────────────────────
print("CLASSIFICATION REPORT — Titanic")
print("=" * 55)
print(classification_report(y_te_t, y_pred_t,
                             target_names=['Not Survived (0)','Survived (1)']))

# ─── ROC-AUC ─────────────────────────────────────────────────
y_prob_t = rf_t.predict_proba(X_te_t)[:, 1]
fpr_t, tpr_t, _ = roc_curve(y_te_t, y_prob_t)
auc_t = auc(fpr_t, tpr_t)

plt.figure(figsize=(7, 5))
plt.plot(fpr_t, tpr_t, 'darkorange', lw=2, label=f'ROC (AUC={auc_t:.3f})')
plt.plot([0,1],[0,1],'navy',lw=2,ls='--',label='Random Classifier')
plt.fill_between(fpr_t, tpr_t, alpha=0.1, color='orange')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve — Titanic Survival', fontsize=12, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"AUC Score: {auc_t:.3f}")

In [ ]:
# ─── Feature Importance ──────────────────────────────────────
fi_t = pd.DataFrame({
    'Feature'   : X_t.columns,
    'Importance': rf_t.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(fi_t['Feature'], fi_t['Importance'],
         color='steelblue', edgecolor='black', linewidth=0.5)
for i, val in enumerate(fi_t['Importance']):
    plt.text(val+0.002, i, f'{val:.3f}', va='center', fontsize=10)
plt.xlabel('Importance');
plt.title('Feature Importance — Titanic Survival', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Most Important Feature: {fi_t.iloc[-1]['Feature']}")
print("→ This is typically 'sex' or 'fare' in the Titanic dataset!")

## 🔧 Step 6: Hyperparameter Tuning

---

> 🎯 **What we learn here**: Use GridSearchCV to find the best hyperparameters for the Titanic model.

In [ ]:
# ═══════════════════════════════════════════════════
# 🔧  GRIDSEARCHCV — TITANIC
# ═══════════════════════════════════════════════════

param_grid_t = {
    'n_estimators': [50, 100, 200],
    'max_depth'   : [None, 5, 8],
    'max_features': ['sqrt', 'log2']
}

gs_t = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_t, cv=5, scoring='accuracy',
    n_jobs=-1, verbose=0
)
gs_t.fit(X_tr_t, y_tr_t)

best_rf_t = gs_t.best_estimator_
y_pred_tuned = best_rf_t.predict(X_te_t)

print("=" * 55)
print("GridSearchCV — Titanic Results")
print("=" * 55)
print("Best Parameters:")
for k, v in gs_t.best_params_.items():
    print(f"  {k:20s}: {v}")

print(f"\nBefore tuning — Test Accuracy: {accuracy_score(y_te_t, y_pred_t)*100:.2f}%")
print(f"After  tuning — Test Accuracy: {accuracy_score(y_te_t, y_pred_tuned)*100:.2f}%")
print(f"Improvement  : {(accuracy_score(y_te_t, y_pred_tuned)-accuracy_score(y_te_t, y_pred_t))*100:+.2f}%")

---

# 🏠 Part 2: Regression — California Housing Price Prediction

---

### 🎯 Problem Statement
> Predict the **median house value** (in $100,000) for California districts based on census data.

### 📊 About the Dataset
- **Source**: California Census Data (1990)
- **Rows**: 20,640 census blocks across California
- **Task**: Regression (predict a number)

| Feature | Description |
|---------|-------------|
| `MedInc` | Median household income in the block |
| `HouseAge` | Median age of houses in the block |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block population |
| `AveOccup` | Average household occupancy |
| `Latitude` | Block latitude |
| `Longitude` | Block longitude |
| `MedHouseVal` | **Target** — Median house value ($100,000s) |

---

> 💡 **Learning Goal**: Handle a larger real dataset with 20,000+ rows and 8 features.

In [ ]:
# ═══════════════════════════════════════════════════
# 📂  LOAD CALIFORNIA HOUSING DATASET
# ═══════════════════════════════════════════════════

housing = fetch_california_housing()
df_ca = pd.DataFrame(housing.data, columns=housing.feature_names)
df_ca['MedHouseVal'] = housing.target  # target in $100,000 units

print("✅ California Housing dataset loaded!")
print(f"Shape: {df_ca.shape}")
print(f"\nFeatures: {list(housing.feature_names)}")
print(f"Target  : MedHouseVal (median house value in $100,000)")
print("\nFirst 5 rows:")
display(df_ca.head())

## 🔍 Step 2: EDA — California Housing

---

> 🎯 **What we learn here**: Explore real numeric data to find which features are most related to house prices.

In [ ]:
# ─── Missing Values & Summary ─────────────────────────────────
print("❓ Missing Values:", df_ca.isnull().sum().sum(), "(none expected)")
print("\n📊 Statistical Summary:")
display(df_ca.describe())

In [ ]:
# ─── Correlation Heatmap ─────────────────────────────────────
plt.figure(figsize=(10, 7))
sns.heatmap(df_ca.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation — California Housing', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

corr = df_ca.corr()['MedHouseVal'].drop('MedHouseVal').abs().sort_values(ascending=False)
print("Features ranked by correlation with MedHouseVal:")
print(corr.to_string())

In [ ]:
# ─── Key Feature Plots ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, feat in enumerate(['MedInc','HouseAge','AveRooms','AveOccup']):
    ax = axes[i//2, i%2]
    ax.scatter(df_ca[feat], df_ca['MedHouseVal'],
               alpha=0.1, color='steelblue', s=5)
    ax.set_xlabel(feat); ax.set_ylabel('MedHouseVal ($100k)')
    ax.set_title(f'{feat} vs House Value', fontweight='bold')

plt.suptitle('Feature vs House Value Scatter Plots', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("💡 MedInc (Median Income) has the STRONGEST positive correlation with house value!")

In [ ]:
# ─── Geographic Plot ─────────────────────────────────────────
plt.figure(figsize=(10, 7))
scatter = plt.scatter(df_ca['Longitude'], df_ca['Latitude'],
                      c=df_ca['MedHouseVal'], cmap='YlOrRd',
                      alpha=0.4, s=5)
plt.colorbar(scatter, label='Median House Value ($100k)')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.title('California Housing — Geographic Distribution\n(Brighter = Higher Value)',
          fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print("💡 High-value areas cluster around the coast (LA, San Francisco)!")

## 🔧 Step 3: Preprocessing — California Housing

---

### 🎯 What We Need to Do

| Task | Why |
|------|-----|
| Split features and target | Standard ML workflow |
| Train-test split | Evaluate on unseen data |
| (Optional) Scale | Random Forest doesn't need it, but good to know |

---

> 💡 **Good News**: California Housing has NO missing values and NO categorical columns — much cleaner than Titanic!

In [ ]:
# ═══════════════════════════════════════════════════
# 🔧  PREPROCESSING — CALIFORNIA HOUSING
# ═══════════════════════════════════════════════════

X_ca = df_ca.drop('MedHouseVal', axis=1)
y_ca = df_ca['MedHouseVal']

X_tr_ca, X_te_ca, y_tr_ca, y_te_ca = train_test_split(
    X_ca, y_ca, test_size=0.2, random_state=42)

print(f"Features (X): {X_ca.shape}")
print(f"Target   (y): {y_ca.shape}")
print(f"\nTrain: {X_tr_ca.shape[0]} rows | Test: {X_te_ca.shape[0]} rows")
print(f"\nTarget stats:")
print(f"  Mean  : ${y_ca.mean()*100000:,.0f}")
print(f"  Median: ${y_ca.median()*100000:,.0f}")
print(f"  Min   : ${y_ca.min()*100000:,.0f}")
print(f"  Max   : ${y_ca.max()*100000:,.0f}")

## 🌳 Step 4: Build Random Forest Regressor

---

> 🎯 **What we learn here**: Apply Random Forest Regression to a large real dataset with 20,000+ rows.

In [ ]:
# ═══════════════════════════════════════════════════
# 🌳  RANDOM FOREST REGRESSOR — CALIFORNIA HOUSING
# ═══════════════════════════════════════════════════

print("Training Random Forest Regressor on California Housing...")
print("(May take 15-30 seconds for 20,000+ rows)\n")

rf_ca = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42,
    n_jobs       = -1   # use all CPU cores for speed
)
rf_ca.fit(X_tr_ca, y_tr_ca)
y_pred_ca = rf_ca.predict(X_te_ca)

mae_ca  = mean_absolute_error(y_te_ca, y_pred_ca)
mse_ca  = mean_squared_error(y_te_ca,  y_pred_ca)
rmse_ca = np.sqrt(mse_ca)
r2_ca   = r2_score(y_te_ca, y_pred_ca)

print("=" * 60)
print("CALIFORNIA HOUSING — RANDOM FOREST REGRESSOR RESULTS")
print("=" * 60)
print(f"MAE  (Mean Absolute Error) : {mae_ca:.4f}   (${mae_ca*100000:,.0f})")
print(f"MSE  (Mean Squared Error)  : {mse_ca:.4f}")
print(f"RMSE (Root MSE)            : {rmse_ca:.4f}  (${rmse_ca*100000:,.0f})")
print(f"R²   Score                 : {r2_ca:.4f}  ({r2_ca*100:.1f}% variance explained)")
print("=" * 60)
print(f"\n💡 On average, our predictions are off by ${mae_ca*100000:,.0f}")
print(f"   The model explains {r2_ca*100:.1f}% of variation in house prices")

In [ ]:
# ─── Evaluation Plots ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
lo = min(y_te_ca.min(), y_pred_ca.min())
hi = max(y_te_ca.max(), y_pred_ca.max())
axes[0].scatter(y_te_ca, y_pred_ca, alpha=0.1, color='steelblue', s=5)
axes[0].plot([lo,hi],[lo,hi],'r--',lw=2,label='Perfect Prediction')
axes[0].set_xlabel('Actual Value ($100k)'); axes[0].set_ylabel('Predicted Value ($100k)')
axes[0].set_title(f'Actual vs Predicted  (R²={r2_ca:.3f})', fontweight='bold')
axes[0].legend()

# Residuals
residuals_ca = y_te_ca - y_pred_ca
axes[1].scatter(y_pred_ca, residuals_ca, alpha=0.1, color='coral', s=5)
axes[1].axhline(0, color='black', lw=2, ls='--')
axes[1].set_xlabel('Predicted ($100k)'); axes[1].set_ylabel('Residual ($100k)')
axes[1].set_title('Residual Plot', fontweight='bold')

plt.suptitle('California Housing — Regression Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Feature Importance ──────────────────────────────────────
fi_ca = pd.DataFrame({
    'Feature'   : X_ca.columns,
    'Importance': rf_ca.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#2ecc71' if x > 0.15 else '#3498db' if x > 0.05 else '#e74c3c'
          for x in fi_ca['Importance']]
plt.barh(fi_ca['Feature'], fi_ca['Importance'],
         color=colors, edgecolor='black', linewidth=0.5)
for i, val in enumerate(fi_ca['Importance']):
    plt.text(val+0.002, i, f'{val:.3f}', va='center', fontsize=10)
plt.xlabel('Importance');
plt.title('Feature Importance — California Housing Regressor', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Most Important Feature: {fi_ca.iloc[-1]['Feature']}")
print("→ MedInc (Median Income) almost always dominates house price prediction!")

## 🔧 Step 5: Hyperparameter Tuning — California Housing

---

> ⚠️ **Note**: GridSearchCV on 20,000 rows can be slow. We use a small param grid here.
> For large datasets, use `RandomizedSearchCV` — it tries RANDOM combinations, much faster!

In [ ]:
# ═══════════════════════════════════════════════════
# 🔧  QUICK HYPERPARAMETER TUNING
# ═══════════════════════════════════════════════════
# Using RandomizedSearchCV for speed on large data

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth'   : [None, 10, 20],
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': [2, 5]
}

rs_ca = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_dist,
    n_iter   = 10,          # Try only 10 random combinations
    cv       = 3,
    scoring  = 'r2',
    n_jobs   = -1,
    random_state=42,
    verbose  = 0
)
rs_ca.fit(X_tr_ca, y_tr_ca)

best_rf_ca   = rs_ca.best_estimator_
y_pred_tuned_ca = best_rf_ca.predict(X_te_ca)
r2_tuned = r2_score(y_te_ca, y_pred_tuned_ca)

print("Best Parameters (RandomizedSearchCV):")
for k, v in rs_ca.best_params_.items():
    print(f"  {k:20s}: {v}")
print(f"\nBefore tuning R²: {r2_ca:.4f}")
print(f"After  tuning R²: {r2_tuned:.4f}")
print(f"Improvement     : {r2_tuned - r2_ca:+.4f}")

---

# 🎓 Capstone Mini-Project

---

### 📋 Your Task

You are a Data Scientist at a bank. The bank wants to predict whether a customer will **default on a loan** (stop paying).

They have given you a dataset to work with. Your job:
1. Load and explore the data
2. Preprocess it (handle nulls, encode)
3. Train a Random Forest Classifier
4. Report: Accuracy, Confusion Matrix, Classification Report, ROC-AUC, Feature Importance

---

### 🎯 Capstone Code — Build it Yourself!

The dataset is generated below. Try to complete the pipeline on your own!

> 💡 **Hint**: Follow the exact same steps we did for Titanic above!

In [ ]:
# ═══════════════════════════════════════════════════
# 🎓  CAPSTONE — LOAN DEFAULT PREDICTION DATASET
# ═══════════════════════════════════════════════════

np.random.seed(99)
n_cap = 1000

df_cap = pd.DataFrame({
    'age'           : np.random.randint(21, 65, n_cap),
    'annual_income' : np.round(np.random.uniform(200000, 2000000, n_cap), -3),
    'loan_amount'   : np.round(np.random.uniform(50000, 500000, n_cap), -3),
    'credit_score'  : np.random.randint(300, 850, n_cap),
    'employment_yrs': np.round(np.random.uniform(0, 30, n_cap), 1),
    'num_prev_loans': np.random.randint(0, 5, n_cap),
    'has_property'  : np.random.randint(0, 2, n_cap),
})

risk = (- 0.5 * df_cap['credit_score'] / 850
        - 0.3 * df_cap['annual_income'] / 2000000
        + 0.4 * df_cap['loan_amount'] / 500000
        + 0.2 * df_cap['num_prev_loans'] / 5)
df_cap['defaulted'] = (risk + np.random.normal(0, 0.15, n_cap) > -0.3).astype(int)

print("🏦 Loan Default Dataset Created!")
print(f"Shape: {df_cap.shape}")
display(df_cap.head())
print(f"\nDefault Rate: {df_cap['defaulted'].mean()*100:.1f}%")
print("\n" + "="*55)
print("YOUR TASK:")
print("  1. Explore the data (EDA)")
print("  2. Train-test split")
print("  3. Train RandomForestClassifier")
print("  4. Show: Confusion Matrix (new way), Classification Report, ROC-AUC")
print("  5. Show Feature Importance")
print("="*55)

In [ ]:
# ─── CAPSTONE: Your solution here ────────────────────────────
# Step 1: EDA (at least check missing values and class balance)

# YOUR CODE HERE


# Step 2: Train-test split (use test_size=0.2, random_state=42)

# YOUR CODE HERE


# Step 3: Train RandomForestClassifier

# YOUR CODE HERE


# Step 4: Evaluation

# YOUR CODE HERE


# Step 5: Feature Importance

# YOUR CODE HERE


print("🎉 If you completed all 5 steps, you are ready for industry!")

In [ ]:
# ─── CAPSTONE SOLUTION (Try on your own first!) ──────────────
# Uncomment and run after you have attempted on your own

# # Step 1: EDA
# print(df_cap.isnull().sum())
# print(df_cap['defaulted'].value_counts())
#
# # Step 2: Split
# X_cap = df_cap.drop('defaulted', axis=1)
# y_cap = df_cap['defaulted']
# X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
#     X_cap, y_cap, test_size=0.2, random_state=42, stratify=y_cap)
#
# # Step 3: Train
# rf_cap = RandomForestClassifier(n_estimators=100, random_state=42)
# rf_cap.fit(X_tr_c, y_tr_c)
# y_pred_c = rf_cap.predict(X_te_c)
#
# # Step 4: Evaluation
# print(f"Accuracy: {accuracy_score(y_te_c, y_pred_c)*100:.2f}%")
# ConfusionMatrixDisplay.from_estimator(rf_cap, X_te_c, y_te_c,
#     display_labels=['No Default','Default'], cmap='Blues')
# plt.title('Loan Default — Confusion Matrix'); plt.show()
# print(classification_report(y_te_c, y_pred_c, target_names=['No Default','Default']))
# y_prob_c = rf_cap.predict_proba(X_te_c)[:, 1]
# print(f"ROC-AUC: {roc_auc_score(y_te_c, y_prob_c):.4f}")
#
# # Step 5: Feature Importance
# fi_cap = pd.DataFrame({'Feature': X_cap.columns, 'Importance': rf_cap.feature_importances_})
# fi_cap = fi_cap.sort_values('Importance', ascending=True)
# plt.barh(fi_cap['Feature'], fi_cap['Importance'], color='steelblue')
# plt.title('Feature Importance — Loan Default'); plt.show()

print("✅ Uncomment the lines above to see the solution!")

---

## 🎓 Summary — What We Did in Notebook 2

---

### Part 1: Titanic Classification

| Step | Action |
|------|--------|
| Load | `sns.load_dataset('titanic')` |
| EDA | Missing values, survival rate by feature, distributions |
| Preprocess | Drop useless cols, fill nulls, LabelEncode categoricals |
| Model | `RandomForestClassifier(n_estimators=100, oob_score=True)` |
| Evaluate | Confusion Matrix (old + new), Classification Report, ROC-AUC |
| Feature Importance | `rf.feature_importances_` |
| Tune | `GridSearchCV` |

---

### Part 2: California Housing Regression

| Step | Action |
|------|--------|
| Load | `fetch_california_housing()` |
| EDA | Correlations, scatter plots, geographic plot |
| Preprocess | No nulls, no categoricals — just split |
| Model | `RandomForestRegressor(n_estimators=100, n_jobs=-1)` |
| Evaluate | MAE, MSE, RMSE, R², Actual vs Predicted, Residuals |
| Tune | `RandomizedSearchCV` (faster for large data) |

---

### Key Difference: GridSearchCV vs RandomizedSearchCV

| | GridSearchCV | RandomizedSearchCV |
|--|---|---|
| Strategy | Try ALL combinations | Try RANDOM N combinations |
| Speed | Slower | Much Faster |
| Use when | Small param grid | Large param grid / big data |

---

> 🚀 **You have now completed the full Random Forest module!**
> You can build, evaluate, and tune Random Forest models on **both synthetic and real data**!